# Gold v2 — MiniCPM greedy 재실행 + 비슷한 정상 이미지 기준선 + 점수 기준선

타당성 점검(2026-10-10) 후속입니다. 기존 결과는 그대로 두고 새 폴더에 씁니다.

1. **검색**: gold·dev 이미지마다 비슷한 정상 이미지를 두 규칙으로 찾음 — `any`(자기 자신만 제외) / `diffdate`(같은 날 촬영 사진도 제외). 날짜별 비율과 "유사도만으로 정상/이상" AUROC를 출력
2. **점수 기준선 (새 추론 없음)**: PatchCore 점수만, 검색 유사도만(임계값은 dev에서 정해 gold에 그대로 적용)
3. **MiniCPM greedy 재실행**: 주요 4조건(원본만 / 고정 정상 / PatchCore 점수 / 결합). 이전 MiniCPM은 temperature 0.7 샘플링이었음
4. **비슷한 정상 이미지 조건**: 4개 모델 × 2규칙(`normal_ref_retrieved`, `normal_ref_retrieved_diffdate`)
5. **모순 지표 다시 계산**: 재실행 결과를 우선 사용(이전 MiniCPM은 `@original`로 보존) → 게시
6. **게시**: `results/latest/gold_v2_main_rerun/`, `results/latest/gold_v2_consistency_metrics/`

GPU: **L4**. 호출 약 534 × (MiniCPM 4조건 + 4모델 × 2조건) ≈ 6,400회, 3~4시간. 20문항마다 Drive 저장, 끊기면 위에서부터 다시 실행하면 이어집니다.

In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
DEV_FOLDER = "lvlm_insplad_qa/gold_v2_dev"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_main_rerun"
REFERENCE_FOLDER = "lvlm_insplad_results/gold_reference_v2"
VX_FOLDER = "lvlm_insplad_results/gold_v2_vision_expert_full"
COMBINED_FOLDER = "lvlm_insplad_results/gold_v2_reference_expert/predictions"
CONSISTENCY_FOLDER = "lvlm_insplad_results/gold_v2_consistency_metrics"

MODELS = ["qwen3_5_4b","internvl3_5_8b","minicpm_v_4_5","llava_onevision_7b"]
MINICPM_RERUN_CONDITIONS = ["image_only","normal_reference","vx:patchcore:score","ref+patchcore:score"]
RETRIEVED_CONDITIONS = ["normal_ref_retrieved","normal_ref_retrieved_diffdate"]
CHECKPOINT_EVERY = 20
PUBLISH = True

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""   # 비우면 자동 검색
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""     # 비우면 자동 검색

In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
DEV_DIR = DRIVE_ROOT / DEV_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)
REFERENCE_DIR = DRIVE_ROOT / REFERENCE_FOLDER
VX_PRED_DIR = DRIVE_ROOT / VX_FOLDER / "eval" / "predictions"
MERGED_MANIFEST = DRIVE_ROOT / VX_FOLDER / "merged_evidence" / "roi_manifest_all.csv"
COMBINED_DIR = DRIVE_ROOT / COMBINED_FOLDER
CONSISTENCY_DIR = DRIVE_ROOT / CONSISTENCY_FOLDER
DINO_CACHE = DEV_DIR / "_dino_cache"   # 정상 후보 임베딩을 dev 세트와 공유

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0",
     "sentencepiece==0.2.2","pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")

In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP); print("UNSUP:", UNSUP_ZIP)

GOLD_QA = sorted(QA_DIR.glob("master_qa_*.jsonl"))[-1]
dev_summary = json.loads((DEV_DIR / "build_summary.json").read_text(encoding="utf-8"))
DEV_QA = DEV_DIR / dev_summary["file"]
for p in (GOLD_QA, DEV_QA, MERGED_MANIFEST):
    print("있음" if p.is_file() else "없음", p)

## 1. 비슷한 정상 이미지 검색 (gold·dev, 두 규칙)

In [ ]:
GOLD_REFS_DIR, DEV_REFS_DIR = RUN_ROOT / "refs_gold", DEV_DIR / "refs"
for qa, out in ((GOLD_QA, GOLD_REFS_DIR), (DEV_QA, DEV_REFS_DIR)):
    run([sys.executable, REPO_DIR / "scripts/build_retrieved_refs.py", "--qa", qa, "--sup-zip", SUP_ZIP,
         "--unsup-zip", UNSUP_ZIP, "--out-dir", out, "--cache-dir", DINO_CACHE])
for name, out in (("gold", GOLD_REFS_DIR), ("dev", DEV_REFS_DIR)):
    diag = json.loads((out / "retrieved_refs_summary.json").read_text(encoding="utf-8"))["diagnostics"]
    print(f"\n[{name}]")
    for rule, d in diag.items():
        print(f"  {rule:9s} 유사도만 AUROC={d['cosine_auroc_for_normal']}  같은 날 비율:",
              {k: v["same_date_rate"] for k, v in d["by_label"].items()})
# 확인: dev 'any' 규칙은 dev 세트를 만들 때 쓴 검색과 같은 규칙이어야 함
dev_imgs = pd.read_csv(DEV_DIR / "dev_images.csv")
dev_refs = pd.read_csv(DEV_REFS_DIR / "retrieved_refs.csv").set_index("image")
same = (dev_imgs.set_index("image").retrieved_reference_image == dev_refs.retrieved_any).mean()
print(f"\ndev 'any' 규칙 = dev 세트 검색 결과 일치율: {same:.3f} (1.000이어야 함)")

## 2. 점수 기준선 (새 추론 없음)

In [ ]:
run([sys.executable, REPO_DIR / "scripts/run_gold_v2_score_baselines.py", "--qa-dir", QA_DIR,
     "--roi-manifest", MERGED_MANIFEST, "--gold-refs-csv", GOLD_REFS_DIR / "retrieved_refs.csv",
     "--dev-refs-csv", DEV_REFS_DIR / "retrieved_refs.csv", "--out-dir", RUN_ROOT])
display(pd.read_csv(RUN_ROOT / "score_baselines.csv").round(3))

## 3·4. MiniCPM greedy 재실행 + 비슷한 정상 이미지 조건 (L4)

In [ ]:
# 끊기면 이 셀만 다시 실행해도 완료된 문항은 재사용
common = ["--qa-dir", QA_DIR, "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP, "--out-dir", RUN_ROOT,
          "--roi-manifest", MERGED_MANIFEST, "--refs-csv", GOLD_REFS_DIR / "retrieved_refs.csv",
          "--checkpoint-every", str(CHECKPOINT_EVERY)]
run([sys.executable, REPO_DIR / "scripts/run_gold_v2_main_conditions.py", *common,
     "--models", "minicpm_v_4_5", "--conditions", *MINICPM_RERUN_CONDITIONS])
run([sys.executable, REPO_DIR / "scripts/run_gold_v2_main_conditions.py", *common,
     "--models", *MODELS, "--conditions", *RETRIEVED_CONDITIONS])
status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
print("row_errors 합계:", sum(s["row_errors"] for s in status), "(0이 아니면 이 셀을 다시 실행)")

## 5. 모순 지표 다시 계산 (재실행 우선, 이전 MiniCPM은 @original)

In [ ]:
run([sys.executable, REPO_DIR / "scripts/run_gold_v2_consistency_metrics.py",
     "--qa-dir", QA_DIR, "--image-only-csv", REPO_DIR / "results/latest/gold_v2_baseline/prediction_review.csv",
     "--reference-dir", REFERENCE_DIR, "--vx-pred-dir", VX_PRED_DIR, "--combined-dir", COMBINED_DIR,
     "--rerun-dir", RUN_ROOT, "--out-dir", CONSISTENCY_DIR, "--models", *MODELS])
cons = pd.read_csv(CONSISTENCY_DIR / "consistency_leaderboard.csv")
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 300)
keep = ["image_only","normal_reference","vx:patchcore:score","ref+patchcore:score","normal_ref_retrieved",
        "normal_ref_retrieved_diffdate"]
cols = ["model_key","condition","q1_anomaly_f1","q2_defect_macro_f1","q3_accuracy","consistency_error",
        "q1q3_contradiction","joint_accuracy","false_defect_rate"]
display(cons[cons.condition.isin(keep)][cols].round(3))
print("\n[MiniCPM: greedy vs 이전 샘플링]")
display(cons[(cons.model_key == "minicpm_v_4_5") & cons.condition.str.replace("@original", "").isin(keep[:4])][cols].round(3))
sig = pd.read_csv(CONSISTENCY_DIR / "significance_main.csv")
print("\n[유의성] delta (CI가 0을 포함하지 않으면 significant)")
display(sig.pivot_table(index=["model_key","baseline","condition"], columns="metric", values="delta").round(3))

## 6. 게시

In [ ]:
if PUBLISH:
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", RUN_ROOT,
         "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_main_rerun",
         "--files", "leaderboard.csv", "run_status.json", "run_metadata.json", "score_baselines.csv"],
        env=os.environ.copy())
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", GOLD_REFS_DIR,
         "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_main_rerun/refs_gold",
         "--files", "retrieved_refs.csv", "retrieved_refs_summary.json"], env=os.environ.copy())
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", DEV_REFS_DIR,
         "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_dev/refs",
         "--files", "retrieved_refs.csv", "retrieved_refs_summary.json"], env=os.environ.copy())
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", CONSISTENCY_DIR,
         "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_consistency_metrics",
         "--files", "consistency_leaderboard.csv", "significance_main.csv", "run_metadata.json"],
        env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_main_rerun/, gold_v2_dev/refs/, gold_v2_consistency_metrics/")